In [10]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:,.4f}'.format)

In [55]:
commodities = pd.read_excel('datos_examen_2.xlsx', sheet_name='Commodities').dropna(axis=1).sort_values(by='Date')
commodities['Date'] = pd.to_datetime(commodities['Date'])
commodities = commodities.set_index('Date')
commodities.head()

,Bid,Ask,Bid.1,Ask.1
Date,,,,
2021-12-15,0.1924,0.1925,1.3345,1.3480
2021-12-16,0.1937,0.1938,1.3725,1.3850
2021-12-17,0.1912,0.1913,1.3855,1.3890
2021-12-20,0.1859,0.1860,1.4180,1.4225
2021-12-21,0.1873,0.1875,1.4070,1.4200


In [21]:
def VaR_APL(df, positions, nc, long):
    bids = [bid for bid in df.columns if 'Bid' in bid]
    asks = [ask for ask in df.columns if 'Ask' in ask]

    last_mid = []

    for i in range(len(bids)):
        df[f'Mid{i}'] = (df[bids[i]] + df[asks[i]]) / 2
        df[f'Spread{i}'] = (df[asks[i]] - df[bids[i]]) / df[f'Mid{i}']
        df[f'Returns{i}'] = df[f'Mid{i}'].pct_change()
        last_mid.append(df[f'Mid{i}'].iloc[-1])

    weights = [weight for weight in (positions * last_mid) / ((last_mid * positions).sum())]
    df['Portfolio_Return'] = np.dot(df[[f'Returns{i}' for i in range(len(bids))]], weights)

    var = np.abs(np.percentile(df['Portfolio_Return'].dropna(), (100-nc)) if long else np.percentile(df['Portfolio_Return'].dropna(), nc))
    cvar = np.abs(df[df['Portfolio_Return'] < -var]['Portfolio_Return'].mean() if long else df[df['Portfolio_Return'] > var]['Portfolio_Return'].mean())

    cl_promedio = np.array([df[col].mean() for col in df.columns if 'Spread' in col])
    cl_estresado = np.array([np.percentile(df[col], 99) for col in df.columns if 'Spread' in col])

    var_apl_promedio, var_apl_estresado = var + cl_promedio.dot(weights), var + cl_estresado.dot(weights)
    cvar_apl_promedio, cvar_apl_estresado = cvar + cl_promedio.dot(weights), cvar + cl_estresado.dot(weights)

    cash_var, cash_cvar = var * (last_mid * positions).sum(), cvar * (last_mid * positions).sum()
    var_apl_promedio_cash, var_apl_estresado_cash = cash_var + cl_promedio.dot(positions*last_mid), cash_var + cl_estresado.dot(positions*last_mid)
    cvar_apl_promedio_cash, cvar_apl_estresado_cash = cl_promedio.dot(positions*last_mid) + cash_cvar, cash_cvar + cl_estresado.dot(positions*last_mid)

    categorias = ['VaR', 'CVaR', 'VaR APL Promedio', 'VaR APL Estresado', 'CVaR APL Promedio', 'CVaR APL Estresado']
    resultados = pd.DataFrame({
        'Métrica': categorias,
        '%': [var, cvar, var_apl_promedio, var_apl_estresado, cvar_apl_promedio, cvar_apl_estresado],
        '$': [cash_var, cash_cvar, var_apl_promedio_cash, var_apl_estresado_cash, cvar_apl_promedio_cash, cvar_apl_estresado_cash]
    })
    return resultados

In [12]:
positions = np.array([30000*15000, 250*112000])
nc = 95
long = False
VaR_APL(commodities, positions, nc, long)

,Métrica,%,$
0,VaR,0.0273,"6,392,309.4393"
1,CVaR,0.0408,"9,557,222.1835"
2,VaR APL Promedio,0.0311,"7,280,695.6981"
3,VaR APL Estresado,0.0381,"8,938,001.4074"
4,CVaR APL Promedio,0.0446,"10,445,608.4423"
5,CVaR APL Estresado,0.0516,"12,102,914.1516"


In [14]:
def VaR_APL(df, positions, nc, long):
    bids = [bid for bid in df.columns if 'Bid' in bid]
    asks = [ask for ask in df.columns if 'Ask' in ask]

    last_mid = []

    for i in range(len(bids)):
        df[f'Mid{i}'] = (df[bids[i]] + df[asks[i]]) / 2
        df[f'Spread{i}'] = (df[asks[i]] - df[bids[i]]) / df[f'Mid{i}']
        df[f'Returns{i}'] = df[f'Mid{i}'].pct_change()
        last_mid.append(df[f'Mid{i}'].iloc[-1])

    weights = [weight for weight in (positions * last_mid) / ((last_mid * positions).sum())]
    df['Portfolio_Return'] = np.dot(df[[f'Returns{i}' for i in range(len(bids))]], weights)

    var = np.abs(np.percentile(df['Portfolio_Return'].dropna(), (100-nc)) if long else np.percentile(df['Portfolio_Return'].dropna(), nc))
    cvar = np.abs(df[df['Portfolio_Return'] < -var]['Portfolio_Return'].mean() if long else df[df['Portfolio_Return'] > var]['Portfolio_Return'].mean())

    cl_promedio = np.array([df[col].mean() for col in df.columns if 'Spread' in col])
    cl_estresado = np.array([np.percentile(df[col], 99) for col in df.columns if 'Spread' in col])

    var_apl_promedio, var_apl_estresado = var + cl_promedio.dot(weights), var + cl_estresado.dot(weights)
    cvar_apl_promedio, cvar_apl_estresado = cvar + cl_promedio.dot(weights), cvar + cl_estresado.dot(weights)

    cash_var, cash_cvar = var * (last_mid * positions).sum(), cvar * (last_mid * positions).sum()
    var_apl_promedio_cash, var_apl_estresado_cash = cash_var + cl_promedio.dot(positions*last_mid), cash_var + cl_estresado.dot(positions*last_mid)
    cvar_apl_promedio_cash, cvar_apl_estresado_cash = cl_promedio.dot(positions*last_mid) + cash_cvar, cash_cvar + cl_estresado.dot(positions*last_mid)

    categorias = ['VaR', 'CVaR', 'VaR APL Promedio', 'VaR APL Estresado', 'CVaR APL Promedio', 'CVaR APL Estresado']
    resultados = pd.DataFrame({
        'Métrica': categorias,
        '%': [var, cvar, var_apl_promedio, var_apl_estresado, cvar_apl_promedio, cvar_apl_estresado],
        '$': [cash_var, cash_cvar, var_apl_promedio_cash, var_apl_estresado_cash, cvar_apl_promedio_cash, cvar_apl_estresado_cash]
    })
    return cl_promedio.dot(positions*last_mid)

VaR_APL(commodities, positions, nc, long)

888386.2588055194

---

In [35]:
bonds = pd.read_excel('datos_examen_2.xlsx', sheet_name='Bonds').dropna(axis=1).sort_values(by='Date')
bonds['Date'] = pd.to_datetime(bonds['Date'])
bonds = bonds.set_index('Date')
bonds.head()

,Bid,Ask,Volume,Bid.1,Ask.1,Volume.1,Bid.2,Ask.2,Volume.2
Date,,,,,,,,,
2019-12-19,128.1719,128.1875,"1,205,409.0000",100.4050,100.4100,"19,631.0000",138.1600,138.1900,"81,465.0000"
2019-12-20,128.2031,128.2344,"806,749.0000",100.4050,100.4100,"23,520.0000",138.2000,138.2400,"79,995.0000"
2019-12-23,128.1094,128.1250,"620,132.0000",100.4050,100.4100,"24,777.0000",137.4800,137.4900,"90,434.0000"
2019-12-24,128.3594,128.3750,"398,672.0000",100.4100,100.4150,"120,733.0000",137.8900,137.9000,"93,855.0000"
2019-12-26,128.4219,128.4375,"402,206.0000",100.4150,100.4200,"35,608.0000",138.3700,138.3800,"74,247.0000"


In [33]:
positions = np.array([12765, 10976, 11764])
nc = 99
long = True
VaR_APL(bonds, positions, nc, long)

array([218.95149956,  57.97180788, 784.94088773])

In [28]:
def VaR_APL(df, positions, nc, long):
    bids = [bid for bid in df.columns if 'Bid' in bid]
    asks = [ask for ask in df.columns if 'Ask' in ask]

    last_mid = []

    for i in range(len(bids)):
        df[f'Mid{i}'] = (df[bids[i]] + df[asks[i]]) / 2
        df[f'Spread{i}'] = (df[asks[i]] - df[bids[i]]) / df[f'Mid{i}']
        df[f'Returns{i}'] = df[f'Mid{i}'].pct_change()
        last_mid.append(df[f'Mid{i}'].iloc[-1])

    weights = [weight for weight in (positions * last_mid) / ((last_mid * positions).sum())]
    df['Portfolio_Return'] = np.dot(df[[f'Returns{i}' for i in range(len(bids))]], weights)

    var = np.abs(np.percentile(df['Portfolio_Return'].dropna(), (100-nc)) if long else np.percentile(df['Portfolio_Return'].dropna(), nc))
    cvar = np.abs(df[df['Portfolio_Return'] < -var]['Portfolio_Return'].mean() if long else df[df['Portfolio_Return'] > var]['Portfolio_Return'].mean())

    cl_promedio = np.array([df[col].mean() for col in df.columns if 'Spread' in col])
    cl_estresado = np.array([np.percentile(df[col], 99) for col in df.columns if 'Spread' in col])

    var_apl_promedio, var_apl_estresado = var + cl_promedio.dot(weights), var + cl_estresado.dot(weights)
    cvar_apl_promedio, cvar_apl_estresado = cvar + cl_promedio.dot(weights), cvar + cl_estresado.dot(weights)

    cash_var, cash_cvar = var * (last_mid * positions).sum(), cvar * (last_mid * positions).sum()
    var_apl_promedio_cash, var_apl_estresado_cash = cash_var + cl_promedio.dot(positions*last_mid), cash_var + cl_estresado.dot(positions*last_mid)
    cvar_apl_promedio_cash, cvar_apl_estresado_cash = cl_promedio.dot(positions*last_mid) + cash_cvar, cash_cvar + cl_estresado.dot(positions*last_mid)

    categorias = ['VaR', 'CVaR', 'VaR APL Promedio', 'VaR APL Estresado', 'CVaR APL Promedio', 'CVaR APL Estresado']
    resultados = pd.DataFrame({
        'Métrica': categorias,
        '%': [var, cvar, var_apl_promedio, var_apl_estresado, cvar_apl_promedio, cvar_apl_estresado],
        '$': [cash_var, cash_cvar, var_apl_promedio_cash, var_apl_estresado_cash, cvar_apl_promedio_cash, cvar_apl_estresado_cash]
    })
    return cl_promedio*positions*last_mid
VaR_APL(bonds, positions, nc, long)

array([218.95149956,  57.97180788, 784.94088773])

Finalizando el ejercicio de bonos, y utilizando los volúmenes proporcionados de los tres activos en tu portafolio para calcular el ADV, responde lo siguiente: ¿Cuántos días te tomaría cerrar completamente la posición en tu portafolio de bonos?

In [ ]:
bonds.head()

,Bid,Ask,Volume,Bid.1,Ask.1,Volume.1,Bid.2,Ask.2,Volume.2
Date,,,,,,,,,
2019-12-19,128.1719,128.1875,"1,205,409.0000",100.4050,100.4100,"19,631.0000",138.1600,138.1900,"81,465.0000"
2019-12-20,128.2031,128.2344,"806,749.0000",100.4050,100.4100,"23,520.0000",138.2000,138.2400,"79,995.0000"
2019-12-23,128.1094,128.1250,"620,132.0000",100.4050,100.4100,"24,777.0000",137.4800,137.4900,"90,434.0000"
2019-12-24,128.3594,128.3750,"398,672.0000",100.4100,100.4150,"120,733.0000",137.8900,137.9000,"93,855.0000"
2019-12-26,128.4219,128.4375,"402,206.0000",100.4150,100.4200,"35,608.0000",138.3700,138.3800,"74,247.0000"


In [40]:
volumen = bonds[[volume for volume in bonds.columns if 'Volume' in volume]]
volumen.head()

,Volume,Volume.1,Volume.2
Date,,,
2019-12-19,"1,205,409.0000","19,631.0000","81,465.0000"
2019-12-20,"806,749.0000","23,520.0000","79,995.0000"
2019-12-23,"620,132.0000","24,777.0000","90,434.0000"
2019-12-24,"398,672.0000","120,733.0000","93,855.0000"
2019-12-26,"402,206.0000","35,608.0000","74,247.0000"


In [46]:
ADV = volumen[-90:].mean()
np.ceil(positions/ADV)

Volume     1.0000
Volume.1   1.0000
Volume.2   1.0000
dtype: float64

In [48]:
import vartools as vt

In [49]:
commodities = pd.read_excel('datos_examen_2.xlsx', sheet_name='Commodities').dropna(axis=1)
commodities['Date'] = pd.to_datetime(commodities['Date'])
commodities = commodities.set_index('Date')
commodities.head()

,Bid,Ask,Bid.1,Ask.1
Date,,,,
2024-12-12,0.2092,0.2094,4.9945,5.0195
2024-12-11,0.2132,0.2133,4.9785,4.9980
2024-12-10,0.2106,0.2107,4.9970,5.0090
2024-12-09,0.2148,0.2149,4.9305,4.9565
2024-12-06,0.2173,0.2175,5.0005,5.0125


In [53]:
positions = np.array([30000*15000, 250*112000])
nc = 0.95
long = False

In [56]:
bonds = pd.read_excel('datos_examen_2.xlsx', sheet_name='Bonds').dropna(axis=1)
bonds['Date'] = pd.to_datetime(bonds['Date'])
bonds = bonds.set_index('Date')
bonds.head()

positions = np.array([12765, 10976, 11764])
nc = 0.99
long = True
vt.var_apl(bonds, positions, nc, long)

,Métrica,Porcentaje,Cash
0,VaR,0.0081,"31,523.2535"
1,VaR Ajustado Promedio,0.0083,"32,585.1177"
2,VaR Ajustado Estresado,0.0110,"43,226.2197"
3,C-VaR,0.0100,"39,171.4385"
4,C-VaR Ajustado Promedio,0.0103,"40,233.3027"
5,C-VaR Ajustado Estresado,0.0130,"50,874.4048"
